# Cell Image Augmentation Notebook

## Overview

This notebook applies data augmentation to labeled microscopy image patches to increase dataset diversity and improve model generalization. The augmentation pipeline applies brightness and contrast adjustments while preserving cell detection labels and binary masks.

### Workflow
1. Load a dataset and select augmentation parameters
2. Initialize output directories for augmented images and labels
3. Iterate through labeled image patches
4. Apply augmentation transformations (brightness, contrast)
5. Validate that cells remain detectable after augmentation
6. Save augmented images, masks, and updated YOLO-format labels

### Output Files
- **Augmented images**: Phase-contrast patches with applied transformations
- **Augmented masks**: Corresponding binary detection masks
- **Augmented labels**: Updated YOLO-format annotations (class, center_x, center_y, width, height)

### Key Features
- Non-destructive augmentation (original data preserved)
- Automatic label bounding box adjustment
- Validation to ensure cells remain visible after transformation
- Comprehensive error handling and progress reporting

### Requirements
- OpenCV (`cv2`)
- NumPy
- Albumentations (augmentation library)
- `CellProcessor` module (custom utilities)

In [18]:
"""Import required libraries for image augmentation and dataset management."""
import cv2
import numpy as np
import os
import albumentations as A
from CellProcessor import (
    use_dataset,
    list_dataset,
    read_yolo_labels
)

## Configuration

Select the dataset to augment and define augmentation parameters.

In [25]:
"""
Define augmentation pipeline using Albumentations.

Transforms:
- RandomBrightnessContrast: Randomly adjust brightness/contrast to simulate lighting variations
  - p=0.2: Apply with 20% probability
  - brightness_limit=0.3: ±30% brightness adjustment
  - contrast_limit=0.15: ±15% contrast adjustment

BboxParams:
- format='yolo': Bounding boxes in normalized YOLO format (center_x, center_y, width, height)
- min_visibility=0.4: Keep boxes only if 40%+ remains visible after transform
- label_fields=['class_labels']: Track class labels during augmentation

additional_targets:
- phase_mask: Also apply transformations to binary detection masks
"""
transform = A.Compose([
    A.RandomBrightnessContrast(
        p=0.2,
        brightness_limit=0.3,
        contrast_limit=0.15
    ),
], 
bbox_params=A.BboxParams(format='yolo', min_visibility=0.4, label_fields=['class_labels']),
additional_targets={'phase_mask': 'mask'}
)

## Load Dataset and Initialize Output Directories

Load dataset configuration and create output directories for augmented images, masks, and labels.

In [20]:
"""List available datasets and select one to use."""
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


In [21]:
"""Load dataset configuration and initialize output directories."""
# Load dataset (using preset 1; modify as needed)
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}\n")

# Construct input paths (source data)
base_output = os.path.join(dataset['Image_path'], dataset['Death_type'])
IMAGES_PHASE_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Phase_Crop/")
LABELED_IMAGES_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Labeled_phase/")
IMAGES_MASKS_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Masks_phase/")

# Construct output paths (augmented data)
AUGMENTED_IMAGES_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Phase_Crop_aug/")
AUGMENTED_MASKS_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Masks_phase_aug/")
AUGMENTED_LABELS_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Labeled_phase_aug/")

# Create output directories
output_dirs = {
    AUGMENTED_IMAGES_DIR_PATH: "Augmented images",
    AUGMENTED_MASKS_DIR_PATH: "Augmented masks",
    AUGMENTED_LABELS_DIR_PATH: "Augmented labels"
}

print("Output directories:")
for dir_path, dir_name in output_dirs.items():
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"  ✓ {dir_name}: {dir_path}")
    except OSError as e:
        print(f"  ✗ {dir_name}: Error creating directory: {e}")

Dataset: MEF cells, Necroptosis death type
Image path: Data

Output directories:
  ✓ Augmented images: Data/Necroptosis/MEF_Phase_Crop_aug/
  ✓ Augmented masks: Data/Necroptosis/MEF_Masks_phase_aug/
  ✓ Augmented labels: Data/Necroptosis/MEF_Labeled_phase_aug/


## Apply Augmentation

For each labeled image patch:
1. Load image, mask, and bounding box labels
2. Apply augmentation pipeline (brightness/contrast adjustments)
3. Update labels if bounding boxes shifted or resized
4. Validate that cells remain visible (min_visibility filter)
5. Save augmented images, masks, and updated labels

In [ ]:
"""
Apply augmentation transformations to all labeled image patches.

For each image:
1. Read phase-contrast image, binary mask, and YOLO labels
2. Apply augmentation pipeline (brightness/contrast)
3. Update labels to reflect bbox adjustments
4. Validate transformed bboxes remain visible
5. Save augmented outputs to corresponding directories
"""

# Precompute file extensions set (faster than tuple in lower().endswith())
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

# Collect source images
image_files = sorted([
    f for f in os.listdir(IMAGES_PHASE_PATH)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS
])
total_images = len(image_files)

if total_images == 0:
    print(f"⚠ No images found in {IMAGES_PHASE_PATH}")
else:
    print(f"Found {total_images} images. Applying augmentation...\n")

# Statistics tracking
processed_count = 0
augmented_count = 0
skipped_count = 0
error_count = 0

# Main augmentation loop (optimized)
for img_idx, image_name in enumerate(image_files, start=1):
    try:
        print(f"[{img_idx}/{total_images}] {image_name}", end=" ... ", flush=True)
        
        # Extract base name once (used multiple times)
        base_name = os.path.splitext(image_name)[0]
        
        # Construct paths (batch together for clarity)
        image_path = os.path.join(IMAGES_PHASE_PATH, image_name)
        label_path = os.path.join(LABELED_IMAGES_DIR_PATH, f"{base_name}.txt")
        mask_path = os.path.join(IMAGES_MASKS_DIR_PATH, image_name)
        
        # Early exit: check label exists before reading images (I/O optimization)
        if not os.path.exists(label_path):
            print("⊘ (no labels)")
            skipped_count += 1
            continue
        
        # Read YOLO labels first (lightweight, fail fast)
        bboxes, class_labels = read_yolo_labels(label_path)
        
        # Early exit: skip if no bboxes (avoids image I/O)
        if not bboxes:
            print("⊘ (no detections)")
            skipped_count += 1
            continue
        
        # Now read images (only if we have valid labels/bboxes)
        img_phase = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
        img_mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        
        # Validate reads
        if img_phase is None or img_mask is None:
            print("✗ (failed to read)")
            error_count += 1
            continue
        
        # Apply augmentation
        transformed = transform(
            image=img_phase,
            bboxes=bboxes,
            class_labels=class_labels,
            phase_mask=img_mask
        )
        
        # Extract transformed data
        transformed_bboxes = transformed['bboxes']
        transformed_class_labels = transformed['class_labels']
        
        # Early exit: skip if all bboxes filtered out by min_visibility
        if not transformed_bboxes:
            print("⊘ (boxes filtered)")
            skipped_count += 1
            continue
        
        # Only unpack image/mask if we're keeping the result (saves memory reference)
        transformed_image = transformed['image']
        transformed_mask = transformed['phase_mask']
        
        # Construct output paths (only needed if saving)
        aug_image_path = os.path.join(AUGMENTED_IMAGES_DIR_PATH, image_name)
        aug_mask_path = os.path.join(AUGMENTED_MASKS_DIR_PATH, image_name)
        aug_label_path = os.path.join(AUGMENTED_LABELS_DIR_PATH, f"{base_name}.txt")
        
        # Batch I/O writes
        cv2.imwrite(aug_image_path, transformed_image)
        cv2.imwrite(aug_mask_path, transformed_mask)
        
        # Generate and write YOLO labels in single operation
        label_lines = [
            f"{int(cls)} {x} {y} {w} {h}\n"
            for cls, (x, y, w, h) in zip(transformed_class_labels, transformed_bboxes)
        ]
        with open(aug_label_path, "w") as f:
            f.writelines(label_lines)
        
        augmented_count += 1
        print(f"✓ ({len(transformed_bboxes)} boxes)")
        processed_count += 1
        
    except Exception as e:
        print(f"✗ Error: {e}")
        error_count += 1

# Print summary
print("\n" + "=" * 60)
print("AUGMENTATION COMPLETE")
print("=" * 60)
print(f"Images processed:        {processed_count}/{total_images}")
print(f"Augmented:               {augmented_count}")
print(f"Skipped (no data):       {skipped_count}")
print(f"Errors:                  {error_count}")
print("=" * 60)
print(f"\nOutput directories:")
print(f"  • Images: {AUGMENTED_IMAGES_DIR_PATH}")
print(f"  • Masks:  {AUGMENTED_MASKS_DIR_PATH}")
print(f"  • Labels: {AUGMENTED_LABELS_DIR_PATH}")